In [14]:
import pandas as pd
import numpy as np

sales = pd.read_csv("Cleaned_sales.csv")
items = pd.read_csv("Cleaned_order_items.csv")
products = pd.read_csv("Cleaned_product_master.csv")
customers = pd.read_csv("Cleaned_customer_master.csv")

print("Sales:", sales.shape)
print("Order items:", items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)

Sales: (138116, 45)
Order items: (397569, 14)
Products: (1175, 10)
Customers: (25000, 12)


In [5]:
print("SALES COLUMNS")
print(sales.columns.tolist())

print("\nORDER ITEMS COLUMNS")
print(items.columns.tolist())

print("\nPRODUCT COLUMNS")
print(products.columns.tolist())

SALES COLUMNS
['order_id', 'order_date', 'order_status', 'sales_channel', 'customer_id', 'customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_type', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_days', 'estimated_delivery_days', 'delivery_status', 'return_status', 'return_reason', 'rating', 'customer_rating', 'review_sentiment', 'marketing_channel', 'campaign_name', 'coupon_code', 'loyalty_points_earned', 'loyalty_points_redeemed', 'quantity', 'gross_sales', 'discount_amount', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'profit', 'profit_margin_percentage', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']

ORDER ITEMS COLUMNS
['order_id', 'product_id', 'quantity', 'unit_price', 'discount_percentage', 'discount_amount', 'gross_sales', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'pr

In [6]:
print("Sales rows per order:")
print(sales.groupby("order_id").size().describe())

print("\nOrder-item rows per order:")
print(items.groupby("order_id").size().describe())

Sales rows per order:
count    138116.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
dtype: float64

Order-item rows per order:
count    138116.000000
mean          2.878515
std           1.512076
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          18.000000
dtype: float64


In [7]:
items["calc_gross_sales"] = (
    items["quantity"] * items["unit_price"]
)

items["calc_discount_amount"] = (
    items["calc_gross_sales"]
    * items["discount_percentage"] / 100
)

items["calc_net_before_tax_shipping"] = (
    items["calc_gross_sales"]
    - items["calc_discount_amount"]
)

display(items[[
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_sales",
    "calc_gross_sales",
    "discount_percentage",
    "discount_amount",
    "calc_discount_amount",
    "net_sales",
    "calc_net_before_tax_shipping"
]].head(10))

,order_id,product_id,quantity,unit_price,gross_sales,calc_gross_sales,discount_percentage,discount_amount,calc_discount_amount,net_sales,calc_net_before_tax_shipping
0,ORD-301242,PROD-000738,2,244.40,488.80,488.80,0.339607,166.00,1.660000,351.88,487.140000
1,ORD-301242,PROD-000181,3,287.13,861.39,861.39,0.362076,311.89,3.118885,593.52,858.271115
2,ORD-773460,PROD-000973,4,754.80,3019.20,3019.20,0.468300,1413.89,14.138901,1919.32,3005.061099
3,ORD-773460,PROD-000268,4,31.36,125.44,125.44,0.336182,42.17,0.421707,99.09,125.018293
4,ORD-449374,PROD-001040,3,6.31,18.93,18.93,0.089678,1.70,0.016976,19.42,18.913024
5,ORD-449374,PROD-000750,2,251.94,503.88,503.88,0.189728,95.60,0.956003,448.93,502.923997
6,ORD-567636,PROD-000639,1,280.19,280.19,280.19,0.016780,4.70,0.047016,301.14,280.142984
7,ORD-567636,PROD-001098,1,264.43,264.43,264.43,0.183924,48.64,0.486351,245.38,263.943649
8,ORD-820028,PROD-000106,4,587.18,2348.72,2348.72,0.051153,120.14,1.201448,2639.24,2347.518552
9,ORD-820028,PROD-000291,1,62.26,62.26,62.26,0.125197,7.79,0.077948,73.92,62.182052


In [8]:
comparison_cols = [
    ("gross_sales", "calc_gross_sales"),
    ("discount_amount", "calc_discount_amount")
]

for actual, calculated in comparison_cols:
    difference = items[actual] - items[calculated]

    print(f"\n{actual} reconciliation")
    print("Total existing:", items[actual].sum())
    print("Total calculated:", items[calculated].sum())
    print("Total difference:", difference.sum())
    print("Rows with difference > 0.02:",
          (difference.abs() > 0.02).sum())


gross_sales reconciliation
Total existing: 206343698.09000003
Total calculated: 206343698.09000003
Total difference: -1.4163248351906077e-10
Rows with difference > 0.02: 0

discount_amount reconciliation
Total existing: 35649166.7
Total calculated: 356491.66155634355
Total difference: 35292675.038443655
Rows with difference > 0.02: 345908


In [9]:
order_totals = (
    items.groupby("order_id", as_index=False)
    .agg(
        calculated_gross_sales=("calc_gross_sales", "sum"),
        calculated_discount=("calc_discount_amount", "sum"),
        recorded_gross_sales=("gross_sales", "sum"),
        recorded_discount=("discount_amount", "sum"),
        recorded_net_sales=("net_sales", "sum"),
        recorded_product_cost=("product_cost", "sum"),
        recorded_profit=("profit", "sum"),
        recorded_tax=("tax_amount", "sum"),
        recorded_shipping=("shipping_cost", "sum")
    )
)

display(order_totals.head())

,order_id,calculated_gross_sales,calculated_discount,recorded_gross_sales,recorded_discount,recorded_net_sales,recorded_product_cost,recorded_profit,recorded_tax,recorded_shipping
0,ORD-100002,1576.81,0.000000,1576.81,0.00,1692.68,822.86,832.79,78.84,37.03
1,ORD-100004,464.88,0.268207,464.88,26.82,533.37,267.98,248.93,78.85,16.46
2,ORD-100016,413.46,0.691894,413.46,69.19,387.31,181.72,186.64,24.09,18.95
3,ORD-100022,1500.75,3.963314,1500.75,396.33,1321.61,710.63,537.37,143.58,73.61
4,ORD-100023,3177.67,10.094403,3177.67,1009.44,2347.64,1691.48,628.53,151.78,27.63


In [10]:
sales_order_totals = (
    sales.groupby("order_id", as_index=False)
    .agg(
        sales_gross_sales=("gross_sales", "sum"),
        sales_discount=("discount_amount", "sum"),
        sales_net_sales=("net_sales", "sum"),
        sales_product_cost=("product_cost", "sum"),
        sales_profit=("profit", "sum")
    )
)

reconciliation = order_totals.merge(
    sales_order_totals,
    on="order_id",
    how="outer",
    indicator=True
)

print(reconciliation["_merge"].value_counts())

display(reconciliation.head(10))

_merge
both          138116
left_only          0
right_only         0
Name: count, dtype: int64


,order_id,calculated_gross_sales,calculated_discount,recorded_gross_sales,recorded_discount,recorded_net_sales,recorded_product_cost,recorded_profit,recorded_tax,recorded_shipping,sales_gross_sales,sales_discount,sales_net_sales,sales_product_cost,sales_profit,_merge
0,ORD-100002,1576.81,0.000000,1576.81,0.00,1692.68,822.86,832.79,78.84,37.03,1576.81,0.00,1692.68,822.86,832.79,both
1,ORD-100004,464.88,0.268207,464.88,26.82,533.37,267.98,248.93,78.85,16.46,464.88,26.82,533.37,267.98,248.93,both
2,ORD-100016,413.46,0.691894,413.46,69.19,387.31,181.72,186.64,24.09,18.95,413.46,69.19,387.31,181.72,186.64,both
3,ORD-100022,1500.75,3.963314,1500.75,396.33,1321.61,710.63,537.37,143.58,73.61,1500.75,396.33,1321.61,710.63,537.37,both
4,ORD-100023,3177.67,10.094403,3177.67,1009.44,2347.64,1691.48,628.53,151.78,27.63,3177.67,1009.44,2347.64,1691.48,628.53,both
5,ORD-100027,34.73,0.029238,34.73,2.92,40.26,11.04,26.81,6.04,2.41,34.73,2.92,40.26,11.04,26.81,both
6,ORD-100033,690.36,0.666137,690.36,66.61,753.03,401.14,347.36,124.75,4.53,690.36,66.61,753.03,401.14,347.36,both
7,ORD-100046,743.04,0.661457,743.04,66.14,777.40,408.62,349.16,80.88,19.62,138.76,20.15,142.04,67.18,59.73,both
8,ORD-100051,3241.75,2.755790,3241.75,275.57,3210.74,1727.00,1446.82,207.64,36.92,3241.75,275.57,3210.74,1727.00,1446.82,both
9,ORD-100052,5508.04,1.651991,5508.04,165.19,6075.73,3322.18,2715.24,694.57,38.31,5508.04,165.19,6075.73,3322.18,2715.24,both


In [11]:
for item_col, sales_col in [
    ("recorded_gross_sales", "sales_gross_sales"),
    ("recorded_discount", "sales_discount"),
    ("recorded_net_sales", "sales_net_sales"),
    ("recorded_product_cost", "sales_product_cost"),
    ("recorded_profit", "sales_profit")
]:
    reconciliation[f"diff_{item_col}"] = (
        reconciliation[item_col] - reconciliation[sales_col]
    )

    print(
        f"{item_col} vs {sales_col}:",
        reconciliation[f"diff_{item_col}"].abs().gt(0.02).sum(),
        "orders differ"
    )

recorded_gross_sales vs sales_gross_sales: 11226 orders differ
recorded_discount vs sales_discount: 9797 orders differ
recorded_net_sales vs sales_net_sales: 11226 orders differ
recorded_product_cost vs sales_product_cost: 11226 orders differ
recorded_profit vs sales_profit: 11226 orders differ


In [12]:
print(sales["order_status"].value_counts(dropna=False))
print("\nReturn status:")
print(sales["return_status"].value_counts(dropna=False))

order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64

Return status:
return_status
NaN         128654
Returned      9462
Name: count, dtype: int64
